##0. Verificacion de librerias




In [ ]:
# Instalación de dependencias (ejecuta solo si es necesario)
# DINOv3 requiere transformers >= 4.56.0
# !pip install "transformers>=4.56.0" torch torchvision pillow requests scikit-learn seaborn supervision ultralytics opencv-python

import importlib

# Lista de paquetes a verificar (nombre de importación -> nombre pip si difiere)
paquetes = [
    ("transformers",  None),
    ("torch",         None),
    ("torchvision",   None),
    ("sklearn",       "scikit-learn"),
    ("PIL",           "pillow"),
    ("cv2",           "opencv-python"),
    ("supervision",   None),
    ("seaborn",       None),
    ("matplotlib",    None),
    ("ultralytics",   None),
    ("requests",      None),
]

todos_ok = True
for nombre_import, nombre_pip in paquetes:
    nombre_pip = nombre_pip or nombre_import
    try:
        mod = importlib.import_module(nombre_import)
        version = getattr(mod, "__version__", "(sin version)")
        print(f"  ok  {nombre_import:<15} {version}")
    except ImportError:
        print(f"  FALTA {nombre_import:<12} -> instala con: pip install {nombre_pip}")
        todos_ok = False

if todos_ok:
    print("\nTodas las dependencias están instaladas.")
else:
    print("\nInstala los paquetes faltantes y vuelve a ejecutar esta celda.")

  ok  transformers    5.10.2
  ok  torch           2.11.0+cu128
  ok  torchvision     0.26.0+cu128
  ok  sklearn         1.6.1
  ok  PIL             11.3.0
  ok  cv2             4.13.0
  FALTA supervision  -> instala con: pip install supervision
  ok  seaborn         0.13.2
  ok  matplotlib      3.10.0
  FALTA ultralytics  -> instala con: pip install ultralytics
  ok  requests        2.32.4

Instala los paquetes faltantes y vuelve a ejecutar esta celda.


In [ ]:
!pip install supervision ultralytics tqdm -q
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.8 MB/s eta 0:00:00


# CLUSTERING DE IMAGENES CON SigLIP

## 1. Autenticacion a HuggingFace

In [ ]:
!hf auth login

Hint: A new version of huggingface_hub (1.19.0) is available! You are using version 1.18.0.
To update, run: hf update

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: n
Token is valid (permission: fineGrained).
The token `Dino` has been saved to /root/.cache/huggingface/stored_tokens
Yo

## 2. Cargar el modelo y verificar que el processor funcione correctamente

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import requests #este era para obtener cosas de internet
from io import BytesIO #Creo que tambien para recibir datos de internet

#1. Verficar si hay GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo seleccionado: {device}")

if (device=="cpu"):
    print("En CPU el modelo tarda un poco")


#2. Cargar el modelo de SigLIP, asi como el procesador

MODELO_ID = "google/siglip-base-patch16-224"
procesador = AutoImageProcessor.from_pretrained(MODELO_ID)
modelo = AutoModel.from_pretrained(MODELO_ID).to(device)

#3. Usar solo en el modo de evaluacion
modelo.train(False) #Usar solamente el modelo de evaluacion

Dispositivo seleccionado: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SiglipModel(
  (text_model): SiglipTextModel(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, 

In [ ]:
#4. Verificar la dimension del embadding con una imagen de prueba

## 3. Obtener la lista de rutas de las imagenes

Cada imagen se guardo en formato tipo:

`#0039-frame_00003510-robot-conf_0.90`


In [ ]:
import os
from tqdm import tqdm
from google.colab import drive

drive.mount("/content/drive")
carpeta_imagenes = "/content/drive/MyDrive/PROYECTO SAM 3/datasets/dataset_cluster_cada30frames_trycatch_videoCompleto"

rutas = []
lista_nombres = os.listdir(carpeta_imagenes)
claves = []
n = len(lista_nombres)


datos = {
    "id": [],
    "frame":[],
    "cluster":[]
}



#Generar la lista de rutas
with tqdm(total = n,desc="Generando rutas de imagenes...",unit="ruta") as pbar:

    for i in range(n):
        rutas.append(os.path.join(carpeta_imagenes,lista_nombres[i]) )
        claves.append(lista_nombres[i][0:20])

        num_id = int(lista_nombres[i][1:5])
        frame = int(lista_nombres[i][13:20])

        datos["id"].append(num_id)
        datos["frame"].append(frame)



        pbar.update(1)


Mounted at /content/drive


Generando rutas de imagenes...: 100%|██████████| 1104/1104 [00:00<00:00, 159491.32ruta/s]


In [ ]:
print(claves[0])

#0001-frame_00001110


In [ ]:
print(len(datos["frame"]))

1104


## 4. Calcular embeddings de las imagenes (493 imagenes de robots)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from collections import Counter

def obtener_embedding(
        ruta_imagen:str,
        procesador: AutoImageProcessor,
        modelo: AutoModel,
        device: str="cpu"
) -> torch.Tensor:

    #1. Cargar imagen desde URL o ruta local
    imagen = Image.open(ruta_imagen,mode="r").convert("RGB")

    #2. Preprocesar: cambiar dimensiones a 224x224, normalizar con media
    # y desviacion estandar de ImageNet -> No se si esto en realidad aplique para SigLIP
    inputs = procesador(images=[imagen],return_tensors="pt").to(device)

    #3. Inferencia sin gradientes
    with torch.no_grad():
        salida = modelo.get_image_features(**inputs)

    #4. pooler_output: vector global que resume toda la imagen
    embedding = salida["pooler_output"]

    #5. Normalizar a norma 1 para que cosine similarity = producto punto
    embedding_normalizado = F.normalize(embedding,dim=-1)

    return embedding_normalizado

In [ ]:
emb = obtener_embedding(rutas[0],procesador,modelo,device)

In [ ]:
print(emb.shape)

torch.Size([1, 768])


In [ ]:
#1. Verificacion de datos minimos

#2. Calcular embeddings para todas las imagenes de clustering
n=len(rutas)
lista_embeddings = []
claves_validas = claves

with tqdm(total = n,desc="Obteniendo embeddings de imagenes...",unit="imagen") as pbar:
    for ruta_imagen in rutas:
        emb = obtener_embedding(ruta_imagen,procesador,modelo,device)
        lista_embeddings.append(emb.squeeze(0).cpu().numpy())
        pbar.update(1)

Obteniendo embeddings de imagenes...: 100%|██████████| 1104/1104 [02:35<00:00,  7.12imagen/s]


## 5. Aplicar K-Means y reduccion de dimensionalidad

In [ ]:
#3. Matriz de embedings [N imagenes, 768]
X = np.array(lista_embeddings)
numero_muestras = len(X)

#4. K-Means con 3 clusters (sabemos que hay 2 o 3 clusters, pero el modelo no lo sabe)
kmeans = KMeans(n_clusters = 3,random_state=0,n_init=60,max_iter=1000)
etiquetas_kmeans = kmeans.fit_predict(X)
datos["cluster"] = etiquetas_kmeans.tolist()

In [ ]:
#5. Proyectar las 768 dimensiones en 2D para poder visualizar
valor_perplexity = min(5,numero_muestras-1)
tsne = TSNE(n_components=2,perplexity=valor_perplexity,random_state=555,n_iter=1000)
X_2D = tsne.fit_transform(X) #[N,2]

print(f"Proyeccion t-SNE completada: {X_2D.shape} (perplexity={valor_perplexity})")


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Proyeccion t-SNE completada: (1104, 2) (perplexity=5)


## 6. Graficar resultados en 2D y en 3D

## 6.2 Graficar resultados en 3D

In [ ]:
colores_cluster   = ["#9b59b6", "#e67e22", "#1abc9c"]

#Panel cluster con K means

In [ ]:
import plotly.express as px
import pandas as pd

# 5. t-SNE: proyectar 768 dimensiones -> 2 dimensiones para visualizar
# perplexity debe ser < n_muestras; usamos min(5,n_muestras-1)
valor_perplexity = min(5,numero_muestras-1)
tsne = TSNE(n_components=3, perplexity = valor_perplexity,random_state = 34, n_iter = 1000)
X_3d = tsne.fit_transform(X) #[N,2]
print(f"Proyeccion t-SNE completada: {X_3d.shape} (perplexity={valor_perplexity})"
)

/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Proyeccion t-SNE completada: (1104, 3) (perplexity=5)


In [ ]:


# 4. Preparar un DataFrame para facilitar la gráfica con Plotly
df_plot = pd.DataFrame(X_3d, columns=['Eje_X', 'Eje_Y', 'Eje_Z'])

# Convertimos las etiquetas numéricas a texto para que Plotly use colores categóricos
df_plot['Cluster'] = ['Cluster ' + str(c) for c in etiquetas_kmeans]


# === PASO 1: Agregamos tu lista 'claves' al DataFrame ===
df_plot['Etiqueta'] = claves


# 5. Generar la gráfica 3D interactiva
fig = px.scatter_3d(
    df_plot,
    x='Eje_X',
    y='Eje_Y',
    z='Eje_Z',
    color='Cluster',
# === PASO 2: Asignamos la columna al hover_name ===
    hover_name='Etiqueta',
    title='Proyección t-SNE en 3D de agrupamiento K-Means (k=3)',
    color_discrete_sequence=px.colors.qualitative.Set1, # Paleta de colores distintiva
    opacity=0.8
)

# Ajustar el tamaño de los puntos para mejor visibilidad
fig.update_traces(marker=dict(size=5))

# Mostrar la gráfica en el navegador
fig.show()

## Exportar el modelo y guardar predicciones en csv

In [ ]:
import joblib

#Exportar el modelo entrenado
#ruta_modelo = "/content/sample_data"
#joblib.dump(kmeans, "kmeans_imagenes_768d.joblib")

['kmeans_imagenes_768d.joblib']

In [ ]:

datos_df = pd.DataFrame(datos)
datos_df = datos_df.sort_values("id")

#ruta_salida_dataframe = "/content/drive/MyDrive/PROYECTO SAM 3/clustering/clustering_3centroides_siglip/dataframe_3cluster_siglip.csv"
#datos_df.to_csv(ruta_salida_dataframe,index=False,sep=",", encoding="utf-8-sig")

## Guardar las predicciones en un dataframe

In [ ]:
#Tabla de conteos
datos_conteo = pd.crosstab(datos_df['id'],datos_df['cluster'])
print("\n--- Tabla 1: Conteo por Cluster ---")
datos_conteo


--- Tabla 1: Conteo por Cluster ---


cluster,0,1,2
id,,,
0,55,2,0
1,24,0,19
2,11,0,0
5,0,4,0
9,16,9,0
...,...,...,...
209,2,12,0
210,16,5,0
212,0,0,3


In [ ]:
#Tabla de porcentajes
datos_porcentaje = pd.crosstab(datos_df['id'], datos_df['cluster'], normalize='index') * 100
print("\n--- Tabla 2: Porcentaje (%) por Cluster ---")
print(datos_porcentaje.round(2)) # Redondeado a 2 decimales para mejor lectura


--- Tabla 2: Porcentaje (%) por Cluster ---
cluster       0       1       2
id                             
0         96.49    3.51    0.00
1         55.81    0.00   44.19
2        100.00    0.00    0.00
5          0.00  100.00    0.00
9         64.00   36.00    0.00
..          ...     ...     ...
209       14.29   85.71    0.00
210       76.19   23.81    0.00
212        0.00    0.00  100.00
216       24.07   74.07    1.85
217        3.45    0.00   96.55

[78 rows x 3 columns]


In [ ]:
datos_porcentaje.round(2)

cluster,0,1,2
id,,,
0,96.49,3.51,0.00
1,55.81,0.00,44.19
2,100.00,0.00,0.00
5,0.00,100.00,0.00
9,64.00,36.00,0.00
...,...,...,...
209,14.29,85.71,0.00
210,76.19,23.81,0.00
212,0.00,0.00,100.00
